# ML Zoomcamp 2026 — Homework 6

Official: `materials/homework.md` (do not edit).
Data: `data/car_fuel_efficiency_2026.csv` (2026 pinned release).
Write-up: `ML_06_HW.md`.

Kernel: this folder `.venv`. Do not use `!pip` / `!wget`.
Submit: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw06


## 0) Imports


In [ ]:
# First align E: cache/temp/thread env, then import libraries used in this homework.

import os
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
# - Names: os.environ.setdefault, numpy/pandas/xgboost,
#   DictVectorizer, root_mean_squared_error, train_test_split,
#   DecisionTreeRegressor, export_text, RandomForestRegressor
# - os.environ.setdefault role: set a default only when the env var is missing.
# - os.environ.setdefault purpose: keep Matplotlib/temp files on the E: cache (not C:)
#   and pin numeric-library threads to 1.
# - DictVectorizer role: turn row dicts into a numeric matrix (incl. one-hot).
# - root_mean_squared_error role: compute RMSE between predictions and targets.
# - DecisionTreeRegressor / RandomForestRegressor / xgb role: tree / forest / boosting regressors.
# - export_text role: show a fitted tree as indented text.
# - Now: prepare paths and tools before loading data.


## 1) Load data and split

Fill missing values with zeros. Split with the exact calls from the homework.


In [ ]:
# Read the local CSV, fill missing values with 0, then split 60/20/20 as the homework states.
# Build train/val matrices with DictVectorizer(sparse=True).

df = pd.read_csv("data/car_fuel_efficiency_2026.csv")
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

del df_train["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]

train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

df.shape, X_train.shape, X_val.shape
# - Names: read_csv, fillna, train_test_split, .values, del,
#   to_dict, DictVectorizer, fit_transform, transform, .shape
# - read_csv role: load the CSV into a DataFrame.
# - fillna(0) role/purpose: fill missing values with 0 to match homework prep rules.
# - train_test_split role: split a table into two chunks by ratio.
# - train_test_split purpose: first hold out 20% test, then split the remaining 80% 75:25
#   to get overall train 60% / val 20% / test 20%.
# - .values role: pull the fuel_efficiency_mpg target column as a NumPy array.
# - del ... fuel_efficiency_mpg purpose: keep the label out of the feature columns.
# - to_dict(orient="records") role: build a list of {column: value} dicts per row.
# - fit_transform role: learn category levels and build the train matrix.
# - transform role: convert validation rows with the same column order only.
# - sparse=True purpose: store a wide one-hot table as a sparse matrix.
# - Now: build X_train / X_val / y_train / y_val for Q1 onward.


## Q1 — Decision tree, max_depth=1

Which original feature is used for splitting? Map a one-hot name such as `origin=USA` back to the original column.

- `vehicle_weight`
- `model_year`
- `origin`
- `fuel_type`

Answer:



In [ ]:
# Fit a depth-1 regression tree and inspect which feature is used for the root split.

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=dv.feature_names_))
# - Names: DecisionTreeRegressor, max_depth, .fit, export_text, feature_names_
# - DecisionTreeRegressor role: build a decision tree for a numeric target.
# - max_depth=1 role/purpose: allow only one question so we see the root-split feature.
# - .fit role: learn split rules from the train matrix and mpg target.
# - export_text role: print the learned rules as human-readable text.
# - feature_names_ role: column-name list produced by DictVectorizer.
# - feature_names_ purpose: map one-hot names like origin=USA back to the original
#   column (origin) when choosing among the answer options.
# - Now: confirm the root split to pick the Q1 answer.


## Q2 — Random forest RMSE

`n_estimators=10`, `random_state=1`. Validation RMSE.

- 0.1837
- 1.837
- 18.37
- 183.7

Answer:



In [ ]:
# Train a 10-tree random forest and compute validation RMSE to match the choices.

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
root_mean_squared_error(y_val, y_pred)
# - Names: RandomForestRegressor, n_estimators, random_state, n_jobs,
# - n_estimators=10 role: put 10 trees in the forest.
# - random_state=1 role/purpose: reproducible randomness matching homework numbers.
# - n_jobs=-1 role: use available CPUs to train in parallel (optional).
# - .predict role: produce mpg predictions on the validation matrix.
# - root_mean_squared_error role: compute RMSE between val targets and predictions.
# - Now: pick the Q2 choice on the right decimal scale.


## Q3 — n_estimators

Try `[10, 50, 100, 150]`, `random_state=1`. Which value has the lowest validation RMSE (3 decimal places)?

- 10
- 50
- 100
- 150

Answer:



In [ ]:
# For each n_estimators candidate, recompute validation RMSE and round to 3 decimals.

scores = []
for n in [10, 50, 100, 150]:
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, model.predict(X_val))
    scores.append((n, round(rmse, 3)))
scores
# - Names: RandomForestRegressor, .fit, .predict,
# - round(rmse, 3) role/purpose: compare at the homework's 3-decimal precision.
# - scores role: collect (n_estimators, rounded RMSE) pairs.
# - Now: pick the n with the lowest validation RMSE for Q3.


## Q4 — max_depth by mean RMSE

`max_depth` in `[10, 15, 20, 25]`. For each, `n_estimators` in `[10, 50, 100, 150]`. `random_state=1`.

- 10
- 15
- 20
- 25

Answer:



In [ ]:
# For each depth, average validation RMSE over four n_estimators values; pick best max_depth.

depth_scores = []
for depth in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        model = RandomForestRegressor(
            n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1
        )
        model.fit(X_train, y_train)
        rmses.append(root_mean_squared_error(y_val, model.predict(X_val)))
    depth_scores.append((depth, round(float(np.mean(rmses)), 3)))
depth_scores
# - Names: max_depth, n_estimators, np.mean, round, float
# - np.mean(rmses) role: average the four RMSE values at that depth.
# - np.mean purpose: choose depth by the homework's mean RMSE.
# - Now: pick the max_depth with the best mean validation RMSE for Q4.


## Q5 — Most important feature

`n_estimators=10`, `max_depth=20`, `random_state=1`. Among `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement`.

Answer:



In [ ]:
# Fit the forest with the given params, then pair feature_importances_ with names to rank features.

rf = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=dv.feature_names_)
importances.sort_values(ascending=False).head(10)
# - Names: feature_importances_, pd.Series, sort_values, head
# - feature_importances_ role: hold per-feature importance from split gain.
# - pd.Series(..., index=feature_names_) role: attach column names to importances.
# - sort_values(ascending=False) role: sort highest importance first.
# - head(10) role: show top 10 so the four answer options are easy to compare.
# - Now: pick the most important original feature for Q5.


## Q6 — XGBoost eta

100 rounds. Compare `eta` 0.3 and 0.1 on validation RMSE.

- 0.3
- 0.1
- Both give equal value

Answer:



In [ ]:
# Wrap train/val as DMatrix, train eta 0.3 and 0.1 for 100 rounds each, compare val RMSE.

features = dv.feature_names_
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, "train"), (dval, "val")]

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

scores = {}
for eta in (0.3, 0.1):
    params = dict(xgb_params)
    params["eta"] = eta
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred = model.predict(dval)
    scores[eta] = root_mean_squared_error(y_val, y_pred)
scores
# - Names: DMatrix, watchlist, xgb_params, eta, objective,
# - DMatrix role: XGBoost container for matrix, labels, and feature names.
# - watchlist role: (data, name) pairs evaluated during training.
# - eta role: learning rate (step size each tree uses to correct error).
# - objective="reg:squarederror" role/purpose: squared-error objective for mpg regression.
# - num_boost_round=100 role: stack 100 trees sequentially.
# - verbose_eval=False role: suppress per-round logs.
# - scores[eta] purpose: compare validation RMSE for the two learning rates side by side.
# - Now: pick the better eta (or a tie) for Q6.
